# Experiment 1: Data Collection

**Objective:** Build a Python data-collection pipeline that gathers weather and air-quality data for selected Bengaluru locations and scrapes book details from a practice bookstore website.

The collected data is saved as CSV files so it can be used later for cleaning, visualization, and modelling.

## Plan

The experiment is split into three parts:

1. Weather data for five Bengaluru locations
2. Air-quality data for the same locations
3. Book title, price, and rating from a paginated practice website

The API sections use Open-Meteo. The scraping section uses Books to Scrape.

In [ ]:
from pathlib import Path
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)

locations = [
    {"name": "MG Road", "lat": 12.9757, "lon": 77.6011},
    {"name": "Whitefield", "lat": 12.9698, "lon": 77.7500},
    {"name": "Jayanagar", "lat": 12.9308, "lon": 77.5838},
    {"name": "Hebbal", "lat": 13.0350, "lon": 77.5970},
    {"name": "Electronic City", "lat": 12.8399, "lon": 77.6770},
]

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AIR_QUALITY_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

session = requests.Session()
session.headers.update({"User-Agent": "data-science-lab/1.0"})

print("Setup complete.")
print("Locations:", ", ".join(place["name"] for place in locations))

## Algorithm

1. Define the five Bengaluru locations with latitude and longitude.
2. Request hourly weather data and keep the latest 21 complete days.
3. Request hourly air-quality data and keep the latest 30 complete days.
4. Scrape each catalogue page from Books to Scrape.
5. Store title, price, and rating for every book.
6. Save the three datasets as CSV files and check their sizes.

## 1. Weather data

For each location, the notebook requests hourly temperature and relative humidity. Only the last 21 complete days are kept, so every location contributes the same number of hourly records.

In [ ]:
weather_frames = []

for place in locations:
    params = {
        "latitude": place["lat"],
        "longitude": place["lon"],
        "hourly": "temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata",
        "past_days": 21,
        "forecast_days": 1,
    }

    response = session.get(WEATHER_URL, params=params, timeout=30)
    response.raise_for_status()
    hourly = response.json()["hourly"]

    df = pd.DataFrame(hourly).rename(
        columns={
            "temperature_2m": "temperature_c",
            "relative_humidity_2m": "humidity_percent",
        }
    )

    df["time"] = pd.to_datetime(df["time"])

    # Keep completed days only. The tail makes the result exactly 21 x 24 rows.
    today = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).normalize()
    df = df[df["time"] < today].tail(21 * 24).copy()

    df.insert(0, "location", place["name"])
    df.insert(1, "latitude", place["lat"])
    df.insert(2, "longitude", place["lon"])

    weather_frames.append(df)
    print(f'{place["name"]}: {len(df)} weather rows')

    time.sleep(0.25)

weather_df = pd.concat(weather_frames, ignore_index=True)

weather_file = OUTPUT_DIR / "bengaluru_weather.csv"
weather_df.to_csv(weather_file, index=False)

print("\nSaved:", weather_file)
print("Shape:", weather_df.shape)
weather_df.head()

## 2. Air-quality data

The same five coordinates are used for PM10, PM2.5, and carbon monoxide. This section keeps 30 complete days of hourly readings.

In [ ]:
air_frames = []

for place in locations:
    params = {
        "latitude": place["lat"],
        "longitude": place["lon"],
        "hourly": "pm10,pm2_5,carbon_monoxide",
        "timezone": "Asia/Kolkata",
        "past_days": 30,
        "forecast_days": 1,
    }

    response = session.get(AIR_QUALITY_URL, params=params, timeout=30)
    response.raise_for_status()
    hourly = response.json()["hourly"]

    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])

    today = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).normalize()
    df = df[df["time"] < today].tail(30 * 24).copy()

    df.insert(0, "location", place["name"])
    df.insert(1, "latitude", place["lat"])
    df.insert(2, "longitude", place["lon"])

    air_frames.append(df)
    print(f'{place["name"]}: {len(df)} air-quality rows')

    time.sleep(0.25)

air_quality_df = pd.concat(air_frames, ignore_index=True)

air_file = OUTPUT_DIR / "bengaluru_air_quality.csv"
air_quality_df.to_csv(air_file, index=False)

print("\nSaved:", air_file)
print("Shape:", air_quality_df.shape)
air_quality_df.head()

In [ ]:
print("Weather date range:")
print(weather_df["time"].min(), "to", weather_df["time"].max())

print("\nAir-quality date range:")
print(air_quality_df["time"].min(), "to", air_quality_df["time"].max())